In [3]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.14.1+cpu
CUDA available: False


In [4]:
# ============================================================
# DAY 22: CUSTOM CNN - SAVE RESULTS
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1. Set project folders
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

MODEL_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

MODEL_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

print("Model folder:", MODEL_DIR)
print("Results folder:", RESULTS_DIR)


# ------------------------------------------------------------
# 2. Check existing Custom CNN model
# ------------------------------------------------------------

model_path = MODEL_DIR / "custom_cnn.pth"

print("\nCustom CNN model exists:", model_path.exists())

if model_path.exists():
    print("Model:", model_path)
else:
    print("Custom CNN model not found.")


# ------------------------------------------------------------
# 3. Enter current Custom CNN results
# ------------------------------------------------------------

results_df = pd.DataFrame([{
    "Model": "Custom CNN",
    "Accuracy": 0.1389,
    "Precision": 0.0000,
    "Recall": 0.0000,
    "F1-Score": 0.0000,
    "ROC-AUC": 0.4700
}])

results_path = RESULTS_DIR / "custom_cnn_results.csv"

results_df.to_csv(
    results_path,
    index=False
)

print("\nPerformance results saved:")
print(results_path)

display(results_df)


# ------------------------------------------------------------
# 4. Save confusion matrix
# ------------------------------------------------------------

cm = [
    [1711, 0],
    [10610, 0]
]

confusion_matrix_df = pd.DataFrame(
    cm,
    index=["Actual Real", "Actual Fake"],
    columns=["Predicted Real", "Predicted Fake"]
)

cm_path = RESULTS_DIR / "custom_cnn_confusion_matrix.csv"

confusion_matrix_df.to_csv(cm_path)

print("\nConfusion matrix saved:")
print(cm_path)

display(confusion_matrix_df)


# ------------------------------------------------------------
# 5. Day 22 completed
# ------------------------------------------------------------

print("\nDay 22 completed successfully.")

Model folder: c:\Users\Raja Kumar\OneDrive\Desktop\DEEP FAKE DETECTION\models
Results folder: c:\Users\Raja Kumar\OneDrive\Desktop\DEEP FAKE DETECTION\results

Custom CNN model exists: False
Custom CNN model not found.

Performance results saved:
c:\Users\Raja Kumar\OneDrive\Desktop\DEEP FAKE DETECTION\results\custom_cnn_results.csv


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Custom CNN,0.1389,0.0,0.0,0.0,0.47



Confusion matrix saved:
c:\Users\Raja Kumar\OneDrive\Desktop\DEEP FAKE DETECTION\results\custom_cnn_confusion_matrix.csv


,Predicted Real,Predicted Fake
Actual Real,1711,0
Actual Fake,10610,0



Day 22 completed successfully.


In [5]:
import torch

print("Torch:", torch.__version__)
print("Torch path:", torch.__file__)

Torch: 2.14.1+cpu
Torch path: c:\Users\Raja Kumar\anaconda3\envs\fastapi_env\Lib\site-packages\torch\__init__.py


In [6]:
import torchvision

print("Torchvision:", torchvision.__version__)
print("Torchvision path:", torchvision.__file__)

Torchvision: 0.29.1+cpu
Torchvision path: c:\Users\Raja Kumar\anaconda3\envs\fastapi_env\Lib\site-packages\torchvision\__init__.py


In [8]:
# ============================================================
# DAY 23: CREATE TRAIN / VAL / TEST DATAFRAMES
# ============================================================

from pathlib import Path
import pandas as pd

def get_video_name(path):
    return Path(str(path)).stem

# Create video names
PROJECT_ROOT = Path.cwd().parent
RESULTS_DIR = PROJECT_ROOT / "results"
face_metadata_df = pd.read_csv(RESULTS_DIR / "final_face_metadata.csv")

if "file_exists" in face_metadata_df.columns:
    face_metadata_df = face_metadata_df[
        face_metadata_df["file_exists"].astype(str).str.lower().eq("true")
    ].copy()

face_metadata_df["video_name"] = face_metadata_df["video_path"].apply(get_video_name)

if "split" in face_metadata_df.columns:
    split_names = face_metadata_df["split"].astype(str).str.lower()
    train_videos = face_metadata_df.loc[split_names == "train", ["video_name"]].drop_duplicates()
    val_videos = face_metadata_df.loc[split_names.isin(["val", "validation"]), ["video_name"]].drop_duplicates()
    test_videos = face_metadata_df.loc[split_names == "test", ["video_name"]].drop_duplicates()
else:
    def load_video_split(*filenames):
        for filename in filenames:
            path = RESULTS_DIR / filename
            if path.exists():
                df = pd.read_csv(path)
                if "video_name" not in df.columns:
                    source_col = next(
                        (col for col in ("video_path", "path", "video") if col in df.columns),
                        None
                    )
                    if source_col is None:
                        raise ValueError(f"No video path/name column found in {path}")
                    df["video_name"] = df[source_col].apply(get_video_name)
                return df
        raise FileNotFoundError(f"Could not find any of: {filenames}")

    train_videos = load_video_split("train_videos.csv")
    val_videos = load_video_split("val_videos.csv", "validation_videos.csv")
    test_videos = load_video_split("test_videos.csv")

train_video_names = set(train_videos["video_name"])
val_video_names = set(val_videos["video_name"])
test_video_names = set(test_videos["video_name"])

# Create train / validation / test dataframes
train_df = face_metadata_df[
    face_metadata_df["video_name"].isin(train_video_names)
].reset_index(drop=True)

val_df = face_metadata_df[
    face_metadata_df["video_name"].isin(val_video_names)
].reset_index(drop=True)

test_df = face_metadata_df[
    face_metadata_df["video_name"].isin(test_video_names)
].reset_index(drop=True)

print("Train samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Test samples:", len(test_df))

Train samples: 39450
Validation samples: 9870
Test samples: 12321


In [9]:

from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent
RESULTS_DIR = ROOT / "results"

# Load the existing metadata and original splits
face_metadata_df = pd.read_csv(
    RESULTS_DIR / "final_face_metadata.csv"
)

# Keep only rows whose face images exist
face_metadata_df = face_metadata_df[
    face_metadata_df["file_exists"].astype(str).str.lower() == "true"
].copy()

# Restore the original train, validation and test splits
train_df = face_metadata_df[
    face_metadata_df["split"] == "train"
].reset_index(drop=True)

val_df = face_metadata_df[
    face_metadata_df["split"] == "val"
].reset_index(drop=True)

test_df = face_metadata_df[
    face_metadata_df["split"] == "test"
].reset_index(drop=True)

print("TRAIN:", len(train_df))
print("VALIDATION:", len(val_df))
print("TEST:", len(test_df))
print("TOTAL:", len(face_metadata_df))

print("\nColumns:", face_metadata_df.columns.tolist())


TRAIN: 39450
VALIDATION: 9870
TEST: 12321
TOTAL: 61641

Columns: ['video_path', 'video_id', 'face_path', 'label', 'class_name', 'file_exists', 'image_size', 'split']


In [10]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent
RESULTS_DIR = ROOT / "results"

print("Project:", ROOT)
print("\nCSV FILES AVAILABLE IN RESULTS:")

for file in RESULTS_DIR.glob("*.csv"):
    try:
        df = pd.read_csv(file, nrows=3)
        print(f"\n{file.name}")
        print("Columns:", list(df.columns))
    except Exception as e:
        print(file.name, "Error:", e)

print("\nEXPECTED FILES:")
for name in ["train_videos.csv", "val_videos.csv", "test_videos.csv",
             "final_face_metadata.csv", "face_metadata.csv"]:
    print(name, "->", (RESULTS_DIR / name).exists())

Project: c:\Users\Raja Kumar\OneDrive\Desktop\DEEP FAKE DETECTION

CSV FILES AVAILABLE IN RESULTS:

cleaned_video_metadata.csv
Columns: ['video_path', 'filename', 'folder', 'fps', 'frame_count', 'width', 'height', 'duration', 'label', 'class_name', 'resolution']

cnn_vs_resnet50_comparison.csv
Columns: ['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']

custom_cnn_confusion_matrix.csv
Columns: ['Unnamed: 0', 'Predicted Real', 'Predicted Fake']

custom_cnn_history.csv
Columns: ['epoch', 'train_loss', 'val_loss']

custom_cnn_results.csv
Columns: ['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']

custom_cnn_results_github.csv
Columns: ['Accuracy', 'Precision', 'Recall', 'F1-score']

face_metadata.csv
Columns: ['video_path', 'video_id', 'face_path', 'label', 'class_name']

final_face_metadata.csv
Columns: ['video_path', 'video_id', 'face_path', 'label', 'class_name', 'file_exists', 'image_size', 'split']

final_frame_metadata.csv
Columns: ['video_path', '

In [11]:
import json
from pathlib import Path

notebook_path = Path.cwd() / "04_model_training_and_comparison.ipynb"
data = json.loads(notebook_path.read_text(encoding="utf-8"))

for i, cell in enumerate(data["cells"]):
    source = "".join(cell.get("source", []))

    if "face_metadata_df" in source or "train_videos" in source:
        print("\n" + "=" * 60)
        print("CELL:", i)
        print(source[:3000])


CELL: 4
# ============================================================
# DAY 23: CREATE TRAIN / VAL / TEST DATAFRAMES
# ============================================================

from pathlib import Path
import pandas as pd

def get_video_name(path):
    return Path(str(path)).stem

# Create video names
PROJECT_ROOT = Path.cwd().parent
RESULTS_DIR = PROJECT_ROOT / "results"
face_metadata_df = pd.read_csv(RESULTS_DIR / "final_face_metadata.csv")

if "file_exists" in face_metadata_df.columns:
    face_metadata_df = face_metadata_df[
        face_metadata_df["file_exists"].astype(str).str.lower().eq("true")
    ].copy()

face_metadata_df["video_name"] = face_metadata_df["video_path"].apply(get_video_name)

if "split" in face_metadata_df.columns:
    split_names = face_metadata_df["split"].astype(str).str.lower()
    train_videos = face_metadata_df.loc[split_names == "train", ["video_name"]].drop_duplicates()
    val_videos = face_metadata_df.loc[split_names.isin(["val", "validation"])

In [12]:
import json
from pathlib import Path

notebook_path = Path.cwd() / "04_model_training_and_comparison.ipynb"
data = json.loads(notebook_path.read_text(encoding="utf-8"))

keywords = [
    "read_csv",
    "face_metadata_df =",
    "train_videos =",
    "val_videos =",
    "test_videos =",
    "train_videos.csv",
    "validation_videos.csv",
    "test_videos.csv"
]

for i, cell in enumerate(data["cells"]):
    source = "".join(cell.get("source", []))

    if any(k.lower() in source.lower() for k in keywords):
        print("\n" + "=" * 60)
        print("CELL:", i)
        print(source[:4000])


CELL: 4
# ============================================================
# DAY 23: CREATE TRAIN / VAL / TEST DATAFRAMES
# ============================================================

from pathlib import Path
import pandas as pd

def get_video_name(path):
    return Path(str(path)).stem

# Create video names
PROJECT_ROOT = Path.cwd().parent
RESULTS_DIR = PROJECT_ROOT / "results"
face_metadata_df = pd.read_csv(RESULTS_DIR / "final_face_metadata.csv")

if "file_exists" in face_metadata_df.columns:
    face_metadata_df = face_metadata_df[
        face_metadata_df["file_exists"].astype(str).str.lower().eq("true")
    ].copy()

face_metadata_df["video_name"] = face_metadata_df["video_path"].apply(get_video_name)

if "split" in face_metadata_df.columns:
    split_names = face_metadata_df["split"].astype(str).str.lower()
    train_videos = face_metadata_df.loc[split_names == "train", ["video_name"]].drop_duplicates()
    val_videos = face_metadata_df.loc[split_names.isin(["val", "validation"])

In [13]:
for i, cell in enumerate(data["cells"]):
    source = "".join(cell.get("source", []))
    if any(k in source.lower() for k in [
        "resnet50",
        "torch.save",
        "optimizer",
        "criterion",
        "num_epochs",
        "epoch"
    ]):
        print("\n" + "=" * 60)
        print("CELL:", i)
        print(source[:6000])


for i, cell in enumerate(data["cells"]):
    source = "".join(cell.get("source", []))
    lines = source.splitlines()
    matches = [
        line for line in lines
        if any(k in line.lower() for k in [
            "optimizer", "backward()", "optimizer.step",
            "for epoch", "torch.save", "state_dict"
        ])
    ]
    if matches:
        print(f"\nCELL {i}:")
        print("\n".join(matches))





CELL: 9
for i, cell in enumerate(data["cells"]):
    source = "".join(cell.get("source", []))
    if any(k in source.lower() for k in [
        "resnet50",
        "torch.save",
        "optimizer",
        "criterion",
        "num_epochs",
        "epoch"
    ]):
        print("\n" + "=" * 60)
        print("CELL:", i)
        print(source[:6000])


for i, cell in enumerate(data["cells"]):
    source = "".join(cell.get("source", []))
    lines = source.splitlines()
    matches = [
        line for line in lines
        if any(k in line.lower() for k in [
            "optimizer", "backward()", "optimizer.step",
            "for epoch", "torch.save", "state_dict"
        ])
    ]
    if matches:
        print(f"\nCELL {i}:")
        print("\n".join(matches))




CELL: 10
from pathlib import Path
import torch
from torchvision import models

ROOT = Path.cwd().parent

MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODEL_DIR / "week4_resnet50.pth"



In [14]:
from pathlib import Path
import torch
from torchvision import models

ROOT = Path.cwd().parent

MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODEL_DIR / "week4_resnet50.pth"

# Save the ResNet50 model; move state_dict tensors to CPU for portability
if "resnet50" not in globals():
    import torch.nn as nn

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    resnet50 = models.resnet50(
        weights=models.ResNet50_Weights.DEFAULT
    )
    resnet50.fc = nn.Linear(resnet50.fc.in_features, 1)
    resnet50 = resnet50.to(device)

    print("ResNet50 initialized for saving.")

torch.save(resnet50.state_dict(), model_path)

# Verify the saved file
if not model_path.exists():
    raise RuntimeError("Model checkpoint could not be saved.")

print("Model saved successfully:", model_path)

Model saved successfully: c:\Users\Raja Kumar\OneDrive\Desktop\DEEP FAKE DETECTION\models\week4_resnet50.pth


In [15]:
# ============================================================
# DAY 24: IMAGE TRANSFORMS
# ============================================================

from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Transforms ready.")

Transforms ready.


In [16]:
# ============================================================
# DAY 24: IMAGE TRANSFORMS
# ============================================================

from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Transforms ready.")

Transforms ready.


In [17]:

from torch.utils.data import Dataset
from PIL import Image
from pathlib import Path
import torch

class DeepfakeDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = Path(row["face_path"])

        if not image_path.is_absolute():
            image_path = ROOT / image_path

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(
            float(row["label"]),
            dtype=torch.float32
        )

        return image, label


In [18]:
# Create datasets

train_dataset = DeepfakeDataset(
    train_df,
    train_transform
)

val_dataset = DeepfakeDataset(
    val_df,
    test_transform
)

test_dataset = DeepfakeDataset(
    test_df,
    test_transform
)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

Train dataset: 39450
Validation dataset: 9870
Test dataset: 12321


In [19]:
# ============================================================
# DAY 23: RESNET50 SETUP
# ============================================================

import torch
import torch.nn as nn
from torchvision import models

# ------------------------------------------------------------
# 1. Check device
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# ------------------------------------------------------------
# 2. Load pretrained ResNet50
# ------------------------------------------------------------

resnet50 = models.resnet50(
    weights=models.ResNet50_Weights.DEFAULT
)


# ------------------------------------------------------------
# 3. Replace final layer for binary classification
# ------------------------------------------------------------

num_features = resnet50.fc.in_features

resnet50.fc = nn.Linear(
    num_features,
    1
)


# ------------------------------------------------------------
# 4. Move model to device
# ------------------------------------------------------------

resnet50 = resnet50.to(device)


# ------------------------------------------------------------
# 5. Check model
# ------------------------------------------------------------

print("\nResNet50 setup completed.")
print("Final layer:", resnet50.fc)

Device: cpu

ResNet50 setup completed.
Final layer: Linear(in_features=2048, out_features=1, bias=True)


In [20]:
# ============================================================
# DAY 24: RESNET50 - TRAINING SETUP
# ============================================================

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import models

# ------------------------------------------------------------
# 1. Device
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# ------------------------------------------------------------
# 2. Load ResNet50
# ------------------------------------------------------------

resnet50 = models.resnet50(
    weights=models.ResNet50_Weights.DEFAULT
)

# Change final layer for Real/Fake classification
num_features = resnet50.fc.in_features

resnet50.fc = nn.Linear(
    num_features,
    1
)

resnet50 = resnet50.to(device)

print("\nResNet50 ready.")


# ------------------------------------------------------------
# 3. Check required datasets
# ------------------------------------------------------------

print("\nChecking datasets...")

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))


# ------------------------------------------------------------
# 4. Create small random subsets for CPU testing
# ------------------------------------------------------------

torch.manual_seed(42)

train_indices = torch.randperm(len(train_dataset))[:100].tolist()
val_indices = torch.randperm(len(val_dataset))[:50].tolist()

train_small = Subset(train_dataset, train_indices)
val_small = Subset(val_dataset, val_indices)


# ------------------------------------------------------------
# 5. Create DataLoaders
# ------------------------------------------------------------

train_small_loader = DataLoader(
    train_small,
    batch_size=16,
    shuffle=True,
    num_workers=0
)

val_small_loader = DataLoader(
    val_small,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

print("Training samples:", len(train_small))
print("Validation samples:", len(val_small))


# ------------------------------------------------------------
# 6. Freeze ResNet50 backbone
# ------------------------------------------------------------

for param in resnet50.parameters():
    param.requires_grad = False

# Train only the final classification layer
for param in resnet50.fc.parameters():
    param.requires_grad = True

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    resnet50.fc.parameters(),
    lr=0.001
)


# ------------------------------------------------------------
# 7. Train for multiple epochs
# ------------------------------------------------------------

epochs = 5

for epoch in range(epochs):

    resnet50.train()
    train_loss = 0.0

    for images, labels in train_small_loader:

        images = images.to(device)
        labels = labels.float().to(device).view(-1, 1)

        optimizer.zero_grad()

        outputs = resnet50(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_small_loader)

    # Validation
    resnet50.eval()
    correct = 0
    total = 0
    val_loss = 0.0

    with torch.no_grad():

        for images, labels in val_small_loader:

            images = images.to(device)
            labels = labels.float().to(device).view(-1, 1)

            outputs = resnet50(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            predictions = (torch.sigmoid(outputs) >= 0.5).float()

            correct += (predictions == labels).sum().item()
            total += labels.numel()

    avg_val_loss = val_loss / len(val_small_loader)
    accuracy = 100 * correct / total

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Train Loss: {avg_train_loss:.4f} "
        f"Val Loss: {avg_val_loss:.4f} "
        f"Accuracy: {accuracy:.2f}%"
    )


# ------------------------------------------------------------
# 8. Save model separately
# ------------------------------------------------------------

torch.save(
    resnet50.state_dict(),
    "resnet50_deepfake_v2.pth"
)

print("\nTraining completed!")
print("Model saved successfully.")





Device: cpu

ResNet50 ready.

Checking datasets...
Train dataset: 39450
Validation dataset: 9870
Training samples: 100
Validation samples: 50
Epoch [1/5] Train Loss: 0.4904 Val Loss: 0.3636 Accuracy: 94.00%
Epoch [2/5] Train Loss: 0.3552 Val Loss: 0.2604 Accuracy: 94.00%
Epoch [3/5] Train Loss: 0.3274 Val Loss: 0.2446 Accuracy: 94.00%
Epoch [4/5] Train Loss: 0.3857 Val Loss: 0.2458 Accuracy: 94.00%
Epoch [5/5] Train Loss: 0.4313 Val Loss: 0.2448 Accuracy: 94.00%

Training completed!
Model saved successfully.


In [33]:

from collections import Counter

print("Validation label distribution:")
print(Counter(
    int(label)
    for _, label in val_small
))

print("\nTraining label distribution:")
print(Counter(
    int(label)
    for _, label in train_small
))


Validation label distribution:
Counter({1: 47, 0: 3})

Training label distribution:
Counter({1: 88, 0: 12})


In [34]:

from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

resnet50.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for images, labels in val_small_loader:

        images = images.to(device)

        outputs = resnet50(images)
        predictions = (torch.sigmoid(outputs) >= 0.5).int()

        y_true.extend(labels.int().numpy())
        y_pred.extend(predictions.cpu().numpy().flatten())

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred, labels=[0, 1]))

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1],
    zero_division=0
))

print("Predicted distribution:", Counter(y_pred))


Confusion Matrix:
[[ 0  3]
 [ 0 47]]

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.94      1.00      0.97        47

    accuracy                           0.94        50
   macro avg       0.47      0.50      0.48        50
weighted avg       0.88      0.94      0.91        50

Predicted distribution: Counter({np.int32(1): 50})


In [21]:
# Freeze initial ResNet50 layers
for param in resnet50.parameters():
    param.requires_grad = False

# Train only the final classification layer
for param in resnet50.fc.parameters():
    param.requires_grad = True

# Update optimizer
optimizer = torch.optim.Adam(
    resnet50.fc.parameters(),
    lr=0.0001
)

print("ResNet50 ready for training.")

ResNet50 ready for training.


In [22]:
from tqdm.auto import tqdm

epochs = 3

for epoch in range(epochs):

    resnet50.train()
    total_loss = 0

    for images, labels in tqdm(train_small_loader):

        images = images.to(device)
        labels = labels.float().to(device).unsqueeze(1)

        optimizer.zero_grad()

        outputs = resnet50(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_small_loader)

    print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}")

print("Training completed!")

  0%|          | 0/7 [00:00<?, ?it/s]

Epoch 1/3 - Loss: 0.3391


  0%|          | 0/7 [00:00<?, ?it/s]

Epoch 2/3 - Loss: 0.4572


  0%|          | 0/7 [00:00<?, ?it/s]

Epoch 3/3 - Loss: 0.4238
Training completed!


In [23]:
test_small = Subset(
    test_dataset,
    range(min(100, len(test_dataset)))
)

test_small_loader = DataLoader(
    test_small,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

In [24]:
# ============================================================
# DAY 26: RESNET50 - BALANCED SMALL TEST EVALUATION
# ============================================================

import torch
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. Create balanced small test set
# ------------------------------------------------------------

real_indices = test_df.index[
    test_df["label"] == 0
].tolist()

fake_indices = test_df.index[
    test_df["label"] == 1
].tolist()

small_indices = (
    real_indices[:50] +
    fake_indices[:50]
)

test_small = Subset(
    test_dataset,
    small_indices
)

test_small_loader = DataLoader(
    test_small,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

print("Test samples:", len(test_small))
print("Real samples:", len(real_indices[:50]))
print("Fake samples:", len(fake_indices[:50]))

# ------------------------------------------------------------
# 2. ResNet50 evaluation
# ------------------------------------------------------------

resnet50.eval()

all_labels = []
all_predictions = []
all_probabilities = []

with torch.no_grad():

    for images, labels in test_small_loader:

        images = images.to(device)

        outputs = resnet50(images)

        probabilities = torch.sigmoid(outputs)

        predictions = (probabilities >= 0.5).int()

        all_labels.extend(labels.numpy())

        all_predictions.extend(
            predictions.cpu().numpy().flatten()
        )

        all_probabilities.extend(
            probabilities.cpu().numpy().flatten()
        )

# ------------------------------------------------------------
# 3. Metrics
# ------------------------------------------------------------

accuracy = accuracy_score(
    all_labels,
    all_predictions
)

precision = precision_score(
    all_labels,
    all_predictions,
    zero_division=0
)

recall = recall_score(
    all_labels,
    all_predictions,
    zero_division=0
)

f1 = f1_score(
    all_labels,
    all_predictions,
    zero_division=0
)

roc_auc = roc_auc_score(
    all_labels,
    all_probabilities
)

cm = confusion_matrix(
    all_labels,
    all_predictions
)

# ------------------------------------------------------------
# 4. Results
# ------------------------------------------------------------

print("\nResNet50 Evaluation Results")
print("=" * 40)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

print("\nConfusion Matrix:")
print(cm)

Test samples: 100
Real samples: 50
Fake samples: 50

ResNet50 Evaluation Results
Accuracy : 0.5000
Precision: 0.5000
Recall   : 1.0000
F1-Score : 0.6667
ROC-AUC  : 0.5340

Confusion Matrix:
[[ 0 50]
 [ 0 50]]


In [25]:
# ============================================================
# DAY 27: CUSTOM CNN vs RESNET50 COMPARISON
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1. Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent
RESULTS_DIR = PROJECT_ROOT / "results"

RESULTS_DIR.mkdir(exist_ok=True)

# ------------------------------------------------------------
# 2. Custom CNN results
# ------------------------------------------------------------

custom_cnn = {
    "Model": "Custom CNN",
    "Accuracy": 0.1389,
    "Precision": 0.0000,
    "Recall": 0.0000,
    "F1-Score": 0.0000,
    "ROC-AUC": 0.4700
}

# ------------------------------------------------------------
# 3. ResNet50 results
# ------------------------------------------------------------

resnet50_result = {
    "Model": "ResNet50",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1-Score": f1,
    "ROC-AUC": roc_auc
}

# ------------------------------------------------------------
# 4. Create comparison table
# ------------------------------------------------------------

comparison_df = pd.DataFrame([
    custom_cnn,
    resnet50_result
])

print("Model Comparison")
print("=" * 60)

display(comparison_df)

# ------------------------------------------------------------
# 5. Save comparison
# ------------------------------------------------------------

comparison_path = RESULTS_DIR / "cnn_vs_resnet50_comparison.csv"

comparison_df.to_csv(
    comparison_path,
    index=False
)

print("\nComparison saved:")
print(comparison_path)

Model Comparison


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Custom CNN,0.1389,0.0,0.0,0.000000,0.470
1,ResNet50,0.5000,0.5,1.0,0.666667,0.534



Comparison saved:
c:\Users\Raja Kumar\OneDrive\Desktop\DEEP FAKE DETECTION\results\cnn_vs_resnet50_comparison.csv


In [26]:
from collections import Counter

print("Actual labels:")
print(Counter(all_labels))

print("\nPredicted labels:")
print(Counter(all_predictions))

Actual labels:
Counter({np.float32(0.0): 50, np.float32(1.0): 50})

Predicted labels:
Counter({np.int32(1): 100})


In [27]:
print("ROC-AUC probabilities:")
print(min(all_probabilities), max(all_probabilities))

ROC-AUC probabilities:
0.65288156 0.99233866


In [28]:
# ============================================================
# DAY 28: WEEK 4 REVIEW & RESULTS ORGANIZATION
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1. Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent
RESULTS_DIR = PROJECT_ROOT / "results"

RESULTS_DIR.mkdir(exist_ok=True)

# ------------------------------------------------------------
# 2. Current ResNet50 preliminary results
# ------------------------------------------------------------

resnet50_results = pd.DataFrame([{
    "Model": "ResNet50",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1-Score": f1,
    "ROC-AUC": roc_auc,
    "Evaluation": "Balanced small test set"
}])

print("ResNet50 Results")
print("=" * 50)
display(resnet50_results)

# ------------------------------------------------------------
# 3. Save ResNet50 results
# ------------------------------------------------------------

resnet50_path = RESULTS_DIR / "resnet50_preliminary_results.csv"

resnet50_results.to_csv(
    resnet50_path,
    index=False
)

print("\nResults saved:")
print(resnet50_path)

# ------------------------------------------------------------
# 4. Save confusion matrix
# ------------------------------------------------------------

cm_df = pd.DataFrame(
    cm,
    index=["Actual Real", "Actual Fake"],
    columns=["Predicted Real", "Predicted Fake"]
)

cm_path = RESULTS_DIR / "resnet50_confusion_matrix.csv"

cm_df.to_csv(cm_path)

print("\nConfusion matrix saved:")
print(cm_path)

display(cm_df)

# ------------------------------------------------------------
# 5. Week 4 summary
# ------------------------------------------------------------

print("\n" + "=" * 50)
print("WEEK 4 SUMMARY")
print("=" * 50)

print("✓ Custom CNN baseline prepared")
print("✓ ResNet50 architecture prepared")
print("✓ ResNet50 training pipeline verified")
print("✓ Balanced test evaluation completed")
print("✓ Accuracy, Precision, Recall and F1 calculated")
print("✓ ROC-AUC calculated")
print("✓ Confusion Matrix generated")
print("✓ Preliminary results saved")

ResNet50 Results


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC,Evaluation
0,ResNet50,0.5,0.5,1.0,0.666667,0.534,Balanced small test set



Results saved:
c:\Users\Raja Kumar\OneDrive\Desktop\DEEP FAKE DETECTION\results\resnet50_preliminary_results.csv

Confusion matrix saved:
c:\Users\Raja Kumar\OneDrive\Desktop\DEEP FAKE DETECTION\results\resnet50_confusion_matrix.csv


,Predicted Real,Predicted Fake
Actual Real,0,50
Actual Fake,0,50



WEEK 4 SUMMARY
✓ Custom CNN baseline prepared
✓ ResNet50 architecture prepared
✓ ResNet50 training pipeline verified
✓ Balanced test evaluation completed
✓ Accuracy, Precision, Recall and F1 calculated
✓ ROC-AUC calculated
✓ Confusion Matrix generated
✓ Preliminary results saved


In [29]:
for i, cell in enumerate(data["cells"]):
    source = "".join(cell.get("source", []))

    if "class DeepfakeDataset" in source:
        print("CELL:", i)
        print(source)

print("".join(data["cells"][13]["source"]))

CELL: 13

from torch.utils.data import Dataset
from PIL import Image
from pathlib import Path
import torch

class DeepfakeDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = Path(row["face_path"])

        if not image_path.is_absolute():
            image_path = ROOT / image_path

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(
            float(row["label"]),
            dtype=torch.float32
        )

        return image, label

CELL: 25
for i, cell in enumerate(data["cells"]):
    source = "".join(cell.get("source", []))

    if "class DeepfakeDataset" in source:
        print("CELL:", i)
        print(source)

print("".join(data["cells"]

In [30]:
source = "".join(data["cells"][13]["source"])

print(source[0:1500])


from torch.utils.data import Dataset
from PIL import Image
from pathlib import Path
import torch

class DeepfakeDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = Path(row["face_path"])

        if not image_path.is_absolute():
            image_path = ROOT / image_path

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(
            float(row["label"]),
            dtype=torch.float32
        )

        return image, label



In [31]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent
RESULTS_DIR = ROOT / "results"

df = pd.read_csv(RESULTS_DIR / "final_face_metadata.csv")

print("Columns:", df.columns.tolist())
print("\nDataset shape:", df.shape)

if "split" in df.columns:
    print("\nSplit distribution:")
    print(df["split"].value_counts(dropna=False))
else:
    print("\nNo 'split' column found.")

Columns: ['video_path', 'video_id', 'face_path', 'label', 'class_name', 'file_exists', 'image_size', 'split']

Dataset shape: (61641, 8)

Split distribution:
split
train    39450
test     12321
val       9870
Name: count, dtype: int64


In [32]:
from pathlib import Path
import torch

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(exist_ok=True)

torch.save(
    resnet50.state_dict(),
    MODEL_DIR / "resnet50_deepfake.pth"
)

print("Model saved successfully!")

Model saved successfully!
